## 실제 데이터 적용 가이드

1. `actual`: 사용자가 실제로 '긍정적'인 리뷰를 남겼거나 플레이 타임이 높은 게임들의 ID/이름 리스트
2. `predicted`: 모델(RAG, 유사도 기반 등)이 사용자에게 추천한 상위 10개 게임 리스트
3. 위 `evaluate_recommendations` 함수에 데이터를 리스트 형태로 묶어서 전달하면 전체 성능을 확인할 수 있습니다.

# 추천 시스템 검증 (Recommendation System Evaluation)

이 노트북은 `final_reviewdata.csv`를 활용하여 추천 모델의 성능을 측정합니다.
데이터를 70:30으로 분할하여 학습(DB 구축)과 테스트(검증)를 진행합니다.

## 지표 설명
1. **Precision@K**: 추천된 K개 중 실제 사용자가 선호하는 아이템의 비율
2. **Recall@K**: 사용자가 선호하는 아이템 중 추천 리스트에 포함된 비율
3. **MRR (Mean Reciprocal Rank)**: 선호 아이템이 상단에 위치할수록 높은 점수

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

def precision_at_k(actual, predicted, k=10):
    if not actual: return 0.0
    predicted = predicted[:k]
    hits = len([p for p in predicted if p in set(actual)])
    return hits / k

def recall_at_k(actual, predicted, k=10):
    if not actual: return 0.0
    predicted = predicted[:k]
    hits = len([p for p in predicted if p in set(actual)])
    return hits / len(actual)

def mrr(actual, predicted):
    if not actual: return 0.0
    for i, p in enumerate(predicted):
        if p in set(actual): return 1 / (i + 1)
    return 0.0

def evaluate_recommendations(test_data, k=10):
    precisions, recalls, mrrs = [], [], []
    for actual, predicted in test_data:
        precisions.append(precision_at_k(actual, predicted, k))
        recalls.append(recall_at_k(actual, predicted, k))
        mrrs.append(mrr(actual, predicted))
    return {f'Precision@{k}': np.mean(precisions), f'Recall@{k}': np.mean(recalls), 'MRR': np.mean(mrrs)}

## 1. 데이터 로드 및 80:20 분할
긍정 리뷰(추천) 데이터를 사용하여 모델을 평가합니다.

In [2]:
# 데이터 로드
df = pd.read_csv('./final_reviewdata.csv')

# '추천' 리뷰만 필터링 (긍정 선호도 기준)
positive_reviews = df[df['추천여부'] == '추천'].copy()

# 70:30 분할 (Train: 추천 DB 구성용, Test: 검증 쿼리용)
train_df, test_df = train_test_split(positive_reviews, test_size=0.2, random_state=42)

print(f"전체 긍정성 리뷰: {len(positive_reviews)}")
print(f"Train(DB): {len(train_df)}, Test(Queries): {len(test_df)}")

전체 긍정성 리뷰: 16916
Train(DB): 13532, Test(Queries): 3384


## 2. 검색/추천 베이스라인 모델 (TF-IDF)
RAG 시스템의 Vector DB 역할을 하는 빠른 베이스라인을 구축합니다.

In [3]:
# 리뷰 텍스트 벡터화
vectorizer = TfidfVectorizer(max_features=5000)
train_matrix = vectorizer.fit_transform(train_df['cleaned_review'].fillna(''))

def get_recommendations(query_text, k=10):
    query_vec = vectorizer.transform([query_text])
    sim = cosine_similarity(query_vec, train_matrix).flatten()
    top_idx = sim.argsort()[-k:][::-1]
    return train_df.iloc[top_idx]['게임명'].tolist()

## 3. 전체 성능 측정
테스트 데이터 샘플에 대해 추천을 수행하고 지표를 산출합니다.

In [ ]:
# 평가용 샘플링 (시간 단축을 위해 200건 추출)
test_samples = test_df.sample(200, random_state=42)
eval_data = []

for _, row in tqdm(test_samples.iterrows(), total=len(test_samples)):
    actual = [row['게임명']]
    query = row['cleaned_review']
    if pd.isna(query): continue
    
    predicted = get_recommendations(query, k=10)
    eval_data.append((actual, predicted))

final_performance = evaluate_recommendations(eval_data, k=10)

print("\n--- [최종 분할 모델 성능 결과] ---")
for metric, value in final_performance.items():
    print(f"{metric}: {value:.4f}")

100%|██████████| 200/200 [00:00<00:00, 239.13it/s]


--- [최종 분할 모델 성능 결과] ---
Precision@10: 0.1440
Recall@10: 1.4400
MRR: 0.2805


In [13]:
import os
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# 수정된 경로
from langchain_chroma import Chroma
from langchain_core.documents import Document
# OpenAI API Key 설정
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# --- [Raw Data 기반 RAG 성능 검증 추가] ---
raw_df = pd.read_csv('../reviewdata/popgame.csv', on_bad_lines='skip', low_memory=False)

# 1. '추천' 리뷰만 필터링하여 raw_positive 정의 (이 부분이 누락되었습니다)
raw_positive = raw_df[raw_df['추천여부'] == '추천'].copy()

# 2. 데이터 분할
raw_train, raw_test = train_test_split(raw_positive, test_size=0.2, random_state=42)

# 3. Raw Data '리뷰 원문'으로 Vector DB 구축
print("Raw Data 기반 Vector DB 생성 중...")
raw_docs = []
# 시간 관계상 1000건만 샘플링하여 DB 구축
for _, row in raw_train.sample(1000, random_state=42).iterrows(): 
    raw_docs.append(Document(
        page_content=str(row['리뷰 원문']),
        metadata={"game_name": row['게임명']}
    ))

raw_vector_db = Chroma.from_documents(raw_docs, OpenAIEmbeddings())

# 4. Raw 기반 추천 함수 
def get_raw_recommendations(query_text, k=10):
    results = raw_vector_db.similarity_search(query_text, k=k)
    return [res.metadata['game_name'] for res in results]

# 5. 성능 측정 실행
test_samples = raw_test.sample(30, random_state=42)
raw_eval_data = []

for _, row in tqdm(test_samples.iterrows(), total=len(test_samples)):
    actual = [row['게임명']]
    query = row['리뷰 원문'] 
    if pd.isna(query): continue
    
    predicted = get_raw_recommendations(query, k=10)
    raw_eval_data.append((actual, predicted))

raw_performance = evaluate_recommendations(raw_eval_data, k=10)

print("\n--- [Raw Data 기반 RAG 성능 결과] ---")
for metric, value in raw_performance.items():
    print(f"{metric}: {value:.4f}")

Raw Data 기반 Vector DB 생성 중...


100%|██████████| 30/30 [00:06<00:00,  4.93it/s]


--- [Raw Data 기반 RAG 성능 결과] ---
Precision@10: 0.0867
Recall@10: 0.8667
MRR: 0.3013
